## **3D Volume Registration and Stitching of Overlapping CT Scans**

**Author:** David Wang Johansen <br>
**Email:** s214743@dtu.dk <br>

### **Introduction**
This notebook shows how to stitch two overlapping volumes using the code developed in the source code.

In the T-rex scans, the individual reconstructions are not aligned only by translation, as the specimen had to be manually moved around, back and forth, and fit inside the FOV of the scanner.

Inter-scan misalignment includes:
- 3D rotations
- Axis flips
- Small isotropic scaling differences

Therefore, stitching requires the use of rigid and similarity transforms.

The registration uses a mutual information metric to optimize the alignment. Thus the alignment maximizes the statistical dependence between intensities of voxels mapped to the same spatial location.

Masking was used on some of the volumes in the pipeline to guide the optimizer to only look at the overlapping regions. This is because some of the scans had very little overlap, which made them more difficult to stitch.

### **Imports**

In [ ]:
import numpy as np
import qim3d.viz

In [ ]:
from src import cfg
from scripts.reconstruct import reconstruct_scans
from scripts.downsample_recons import downsample_recons
from src.io import load_recon
from src.volume_stitcher import estimate_transform, stitch

### **Reconstruction the scans**

Available is two scans diagonal from each other in layer 4. These were not paired in the pipeline, but are used here since they are more challenging, and since the full dataset is not made publicly available.

In [ ]:
scans = [cfg.SCANS_BY_LAYER[4][2], cfg.SCANS_BY_LAYER[4][0]]
print(scans)

We reconstruct the scans using the padding method described in the `padding.ipynb` notebook.

In [ ]:
reconstruct_scans(scans=scans, dataset=cfg.full_res)

Due to the stitching being quite time consuming on the full scans, we here downsample the reconstructions and use those instead.

In [ ]:
downsample_recons(scans=scans)

### **Estimating the transform**

In [ ]:
vol_fixed = load_recon(scans[0], dataset=cfg.downsampled)
vol_moving = load_recon(scans[1], dataset=cfg.downsampled)

We start by visualizing the two volumes (slide between them), to see what we are dealing with.

In [ ]:
qim3d.viz.overlay(vol_fixed, vol_moving)

As we can see, the non-overlapping areas are quite large. The registration method uses a metric called Mattes Mutual Information, see https://antspy.readthedocs.io/en/latest/registration.html for more details about the registration. This measures the statistical dependence between corresponding pixels in the two volumes. This is highly prone to converge towards local optima, hence in some cases we need to put a prior on where to sample these pixels from such that we find our desired optimum. We do this by masking out the irrelevant (non-overlapping) parts. This does not need to be very precise, and in the following we choose some very rough estimates, which still get the job done.

In [ ]:
# here we mask out 50% in two of the axes
s = vol_fixed.shape
mask_fixed = np.zeros(s, dtype=bool)
mask_fixed[:, int(0.5*s[1]):, int(0.5*s[2]):] = 1
qim3d.viz.slicer(vol_fixed*mask_fixed, colormap='grey', image_size=4)

In [ ]:
# we do a similar thing for the other volume
s = vol_moving.shape
mask_moving = np.zeros(s, dtype=bool)
mask_moving[:, :int(0.5*s[1]), :int(0.5*s[2])] = 1

qim3d.viz.slicer(vol_moving*mask_moving, colormap='grey', image_size=4)

Now we pass these masked volumes into the registration step. We restrict the transforms those of the rigid type, which means that only translations and rotations will be considered.

In [ ]:
A_hom = estimate_transform(vol_fixed*mask_fixed, vol_moving*mask_moving, type_of_transform='Rigid')

The output is a 4x4 homogeneous matrix representing the forward transform:

In [ ]:
print(A_hom)

It can be seen that there is a non-trivial translation in each coordinate and also a non-trivial rotation.

### **Stitching the volumes**

Great, now we have the transform that tells us how to geometrically map one volume into the other. Next, the stitching is done to merge the two volumes together into one larger volume. We also include some more outputs with `debug=True` to visualize what is going. The parameter `approximate` makes the stitching much faster and much lower memory traded off for slightly lower accuracy (in the interpolation algorithms). In this case the difference in quality is barely (if at all) noticeable.

In [ ]:
vol_stitched, canvas1, canvas2 = stitch(vol_fixed, vol_moving, A_hom, approximate=True, debug=True)

See the alignment by sliding between each original volume transformed to the same coordinate system:

In [ ]:
qim3d.viz.overlay(canvas1, canvas2)

<figure>
    <img src="../assets/stitching_overlay_comparison.png" width="800">
    <figcaption>
        <em> Preview of sliding between the two volumes in the above code block. </em>
    </figcaption>
</figure>

See the resulting stitch:

In [ ]:
qim3d.viz.slicer(vol_stitched, colormap='grey', image_size=6, min_value=0, max_value=np.quantile(vol_stitched, 0.9975))

<figure>
    <img src="../assets/stitching_result.png" width="600">
    <figcaption>
        <em> Preview of running the above code block. </em>
    </figcaption>
</figure>

It can be seen that both a translation and a smaller rotation was needed to do the alignment. We blend the overlapping regions by doing a weighted average depending on how far out the pixels are along each volume. The result is that the overlapping regions are seamless. We see that we still miss parts from the other 2 diagonals, which would complete the layer.

In the pipeline, this stitching is done for on pairwise volumes in the steps inside of `scripts/stitch_pipeline.py`. Some of the steps depend on other steps. For example, to merge layer 4 and 3 we first need to stitch all the scans inside of the individual layers.

For visualizing the final full resolution stitched result with all the scans (as in the 3D renderings), the volume intensities were clipped at negative values and values above the 99.75\% percentile. Since some of the supporting platform (during acquisition in the CT scanner) was visible at the bottom these were also cropped away. This was easy to do as it was far away from the object. It is however visible in the bottom of the stitch above.